In [ ]:
import sys
from pathlib import Path
import torch

SRC = str((Path('..') / 'src').resolve())
sys.path.insert(0, SRC)

from layer import L_DG, L_CA3

## Step 3: L_DG — Dentate Gyrus

**Role**: Pattern separation. Maps ECin's distributed input to a highly sparse (~1% active)
orthogonal representation, so that similar ECin patterns produce distinct DG codes.

**Key parameters (Schapiro 2017 §2.a.iii)**:
- `ecin_frac = 0.25`: each DG unit receives from 25% of ECin units
- `k_frac = 0.01`: ~1% of DG units active after kWTA
- No recurrent connections (unlike CA3)

**Understanding check**: Why 1% sparsity?  
→ Forces near-orthogonal DG codes per episode. CA3 then stores distinct attractors per episode.
If sparsity were 50% (like CA3), DG patterns would heavily overlap → CA3 attractors interfere → pattern completion fails.

In [ ]:
# Instantiate and inspect structure
# n_input=15 (Schapiro 2017 n_items); n_DG=100
dg = L_DG(n_input=15, n_DG=100, k_frac=0.01, ecin_frac=0.25, use_euler=True)

print(f"W shape  : {dg.W.shape}")           # (15, 100): each column = one DG unit
print(f"mask shape: {dg.mask.shape}")        # (15, 100)
print(f"mask density: {dg.mask.mean():.3f}") # ~0.25 (25% connectivity)
print(f"n_active target: {max(1, int(dg.k_frac * dg.n_DG))}")  # 1 unit at k_frac=0.01

In [ ]:
# Set random weights to make forward pass non-trivial
# (at init all weights are zero → nxx1(0) ≈ 0 → kWTA keeps top-1 arbitrarily)
torch.manual_seed(42)
dg.W.data = torch.randn(15, 100) * 0.1

# Forward pass: single ECin pattern (item 0 active at 1.0)
a_ecin = torch.zeros(15)
a_ecin[0] = 1.0

dg.reset()
act = dg(a_ecin)

n_active = (act > 0).sum().item()
print(f"Active DG units: {n_active} / {dg.n_DG} = {n_active / dg.n_DG:.3f}")
print(f"Max activity   : {act.max().item():.4f}")
print(f"Active indices : {(act > 0).nonzero(as_tuple=True)[0].tolist()}")

In [ ]:
# Pattern separation: two similar ECin patterns → very different DG codes
# Item A: units 0 and 1 active (moving window: curr=1.0, prev=0.9)
a_A = torch.zeros(15); a_A[0] = 1.0; a_A[1] = 0.9
# Item B: units 1 and 2 active (shares unit 1 with A)
a_B = torch.zeros(15); a_B[1] = 1.0; a_B[2] = 0.9

cos_ecin = torch.nn.functional.cosine_similarity(a_A.unsqueeze(0), a_B.unsqueeze(0)).item()

dg.reset(); act_A = dg(a_A).clone()
dg.reset(); act_B = dg(a_B).clone()

cos_dg = torch.nn.functional.cosine_similarity(act_A.unsqueeze(0), act_B.unsqueeze(0)).item()

print(f"ECin cosine similarity (A, B): {cos_ecin:.3f}")
print(f"DG   cosine similarity (A, B): {cos_dg:.3f}")
print("Pattern separation: DG similarity should be much lower than ECin similarity.")

In [ ]:
# CHL weight update — masked
dg.reset(); act_minus = dg(a_A).clone()
dg.reset(); act_plus  = dg(a_B).clone()

W_before = dg.W.data.clone()
dg.update_weights(a_ECin_minus=a_A, a_ECin_plus=a_B,
                  a_DG_minus=act_minus, a_DG_plus=act_plus, lr=0.4)
delta_W = dg.W.data - W_before

print(f"Weights changed at masked positions  : {(delta_W[dg.mask.bool()] != 0).sum().item()}")
print(f"Weights changed at unmasked positions: {(delta_W[~dg.mask.bool()] != 0).sum().item()}")
print("Unmasked positions must remain zero.")

---
## Step 4: L_CA3 — CA3 Field

**Role**: Pattern completion via recurrent attractor dynamics.
A partial cue from DG activates a partial CA3 pattern; recurrent connections reinstate the full stored pattern.

**Key parameters (Schapiro 2017 §2.a.iii)**:
- `dg_frac = 0.05`: 5% sparse mossy fibre from DG (much sparser than ECin→DG)
- `k_frac = 0.10`: ~10% active — less sparse than DG to allow overlap for completion
- `W_rec`: fully connected CA3→CA3 (Hopfield attractor)

**Understanding check**: What does W_rec do on the first trial of a new item?  
→ W_rec starts at zero → no recurrent input → CA3 settles based on DG input alone (feedforward only).
After the first CHL update W_rec begins to store the pattern. After several trials the attractor is stable enough that a partial DG cue reinstates the full CA3 pattern.

In [ ]:
# Instantiate and inspect structure
ca3 = L_CA3(n_DG=100, n_CA3=50, k_frac=0.10, dg_frac=0.05, use_euler=True)

print(f"W_ff shape    : {ca3.W_ff.shape}")         # (100, 50)
print(f"mask_ff shape : {ca3.mask_ff.shape}")       # (100, 50)
print(f"mask_ff density: {ca3.mask_ff.mean():.3f}") # ~0.05 (5% mossy fibre)
print(f"W_rec shape   : {ca3.W_rec.shape}")         # (50, 50) — fully connected
print(f"n_active target: {max(1, int(ca3.k_frac * ca3.n_CA3))}")  # 5 units at k_frac=0.10

In [ ]:
# Forward pass — sparsity check
torch.manual_seed(0)
ca3.W_ff.data  = torch.randn(100, 50) * 0.1
ca3.W_rec.data = torch.randn(50, 50)  * 0.05

# Use DG output from item A above
a_dg = act_A.clone()  # from L_DG cell above

ca3.reset()
act_ca3 = ca3(a_dg)

n_active = (act_ca3 > 0).sum().item()
print(f"Active CA3 units: {n_active} / {ca3.n_CA3} = {n_active / ca3.n_CA3:.3f}")
print(f"Expected ~10% = {int(ca3.k_frac * ca3.n_CA3)} units")

In [ ]:
# Recurrent dynamics: activity evolves over settling cycles
# After storing a pattern, partial input should reinstate the full pattern

# Step 1: store pattern by running full DG input for 25 cycles (Q1)
ca3.reset()
for _ in range(25):
    stored = ca3(a_dg)
stored = stored.clone()
print(f"Stored pattern active units: {(stored > 0).sum().item()}")

# Step 2: do one CHL update to write pattern into W_rec
# (minus=zeros, plus=stored — simplified; real training uses ActM vs ActP)
ca3.update_weights(
    a_DG_minus=torch.zeros(100), a_DG_plus=a_dg,
    a_CA3_minus=torch.zeros(50), a_CA3_plus=stored,
    lr=0.4,
)

# Step 3: present zero DG input — recurrent W_rec alone should partially reinstate
ca3.reset()
act_from_recurrent = ca3(torch.zeros(100))  # no DG input
overlap = ((stored > 0) & (act_from_recurrent > 0)).sum().item()
print(f"Pattern completion from zero DG input:")
print(f"  active in stored  : {(stored > 0).sum().item()}")
print(f"  active in recalled: {(act_from_recurrent > 0).sum().item()}")
print(f"  overlap           : {overlap}")
print("(W_rec starts near zero; overlap increases with more CHL updates)")

In [ ]:
# CHL weight update — check mask for W_ff, no mask for W_rec
ca3_test = L_CA3(n_DG=100, n_CA3=50, k_frac=0.10, dg_frac=0.05)
torch.manual_seed(1)
ca3_test.W_ff.data  = torch.randn(100, 50) * 0.1
ca3_test.W_rec.data = torch.randn(50, 50)  * 0.05

a_dg_m = act_A.clone()
a_dg_p = act_B.clone()
ca3_test.reset(); a_ca3_m = ca3_test(a_dg_m).clone()
ca3_test.reset(); a_ca3_p = ca3_test(a_dg_p).clone()

Wff_before  = ca3_test.W_ff.data.clone()
Wrec_before = ca3_test.W_rec.data.clone()

ca3_test.update_weights(a_dg_m, a_dg_p, a_ca3_m, a_ca3_p, lr=0.4)

dWff  = ca3_test.W_ff.data  - Wff_before
dWrec = ca3_test.W_rec.data - Wrec_before

print(f"W_ff  changed at masked   positions: {(dWff[ca3_test.mask_ff.bool()] != 0).sum().item()}")
print(f"W_ff  changed at unmasked positions: {(dWff[~ca3_test.mask_ff.bool()] != 0).sum().item()}")
print(f"W_rec changed (all {ca3_test.n_CA3}×{ca3_test.n_CA3} = {ca3_test.n_CA3**2} entries): {(dWrec != 0).sum().item()}")
print("W_ff unmasked must stay zero. W_rec fully updated (no mask).")